# Data Check: Trade Derived

Quick visual check for trade-derived pipeline data quality.

1. Run the trade_derived pipeline steps to populate parquet data
2. Load each data type and plot for visual inspection

Steps:
- **PopulateCandleBarStep** (1-min candle bars)
- **PopulateHeikinAshiBarStep** (1-min Heikin Ashi bars)
- **PopulateRangeBarStep** (range bars)
- **PopulateAggressionStep** (trade aggression metrics)

## Setup

In [ ]:
import sys

sys.path.insert(0, "../../src")

import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from qate.core.model import ExchangeName, Market
from qate.core.symbol import Symbol
from qate.util.dt_range import DtRange

from makalu import settings as sys_env
from makalu.storage.parquet_store import ParquetStore
from pipeline.market_pair import EXECUTION_SIZE

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
%matplotlib inline

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s"
)

POPULATE_DATA = False

MARKET = Market(ExchangeName.BINANCE, Symbol.XRP_USDT)
DATA_START_DATE = "20251110"
DATA_END_DATE = "20251111"

DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)
EXECUTION_SIZE_STR = EXECUTION_SIZE.get(MARKET.symbol)

print(f"Market:         {MARKET.id}")
print(f"Date range:     {DATA_START_DATE} - {DATA_END_DATE}")
print(f"Execution size: {EXECUTION_SIZE_STR}")

## 1. Run Pipeline Steps

In [ ]:
from pipeline.framework import Pipeline
from pipeline.stages import TradeDerivedStage
from pipeline.trade_derived_steps import (
    PopulateAggressionStep,
    PopulateCandleBarStep,
    PopulateHeikinAshiBarStep,
    PopulateRangeBarStep,
)


def populate_data():
    parquet_store = ParquetStore(sys_env.get_parquet_root_dir())

    pipeline = Pipeline(parquet_store)
    pipeline.stage(
        TradeDerivedStage(MARKET)
        .step(PopulateCandleBarStep(interval_seconds=60))
        .step(PopulateHeikinAshiBarStep(interval_seconds=60))
        .step(PopulateRangeBarStep(box_size_rate=0.001))
        .step(PopulateAggressionStep(windows_sec=[1, 3, 5]))
    )

    print("Running pipeline...")
    pipeline.process(DT_RANGE)
    print("Done.")


if POPULATE_DATA:
    populate_data()

## 2. Candle Bars (1-min)

In [ ]:
store = ParquetStore(sys_env.get_parquet_root_dir())
candle_df = store.create_chart(MARKET, "CANDLE").read(DT_RANGE)

print(f"Shape: {candle_df.shape}")
print(f"Time range: {candle_df.index.min()} ~ {candle_df.index.max()}")
print(f"Columns: {list(candle_df.columns)}")
print(f"NaN counts:\n{candle_df.isna().sum()}")
display(candle_df.describe())
display(candle_df.head())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

axes[0].plot(candle_df.index, candle_df["close"], linewidth=0.8, label="Close")
axes[0].fill_between(
    candle_df.index, candle_df["low"], candle_df["high"], alpha=0.2, label="High-Low"
)
axes[0].set_ylabel("Price")
axes[0].set_title(f"{MARKET.id} Candle Bars (1-min)")
axes[0].legend()

axes[1].bar(
    candle_df.index, candle_df["volume"], width=0.0005, alpha=0.7, color="steelblue"
)
axes[1].set_ylabel("Volume")

axes[2].plot(candle_df.index, candle_df["tick_count"], linewidth=0.5, color="purple")
axes[2].set_ylabel("Tick Count")
axes[2].set_xlabel("Time")

plt.tight_layout()
plt.show()

## 3. Heikin Ashi Bars (1-min)

In [ ]:
ha_df = store.create_chart(MARKET, "HEIKIN_ASHI").read(DT_RANGE)

print(f"Shape: {ha_df.shape}")
print(f"Time range: {ha_df.index.min()} ~ {ha_df.index.max()}")
print(f"NaN counts:\n{ha_df.isna().sum()}")
display(ha_df.describe())
display(ha_df.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Color by direction
colors = ["green" if c >= o else "red" for o, c in zip(ha_df["open"], ha_df["close"])]
axes[0].bar(
    range(len(ha_df)),
    ha_df["close"] - ha_df["open"],
    bottom=ha_df["open"],
    color=colors,
    alpha=0.7,
    width=1.0,
)
axes[0].vlines(
    range(len(ha_df)), ha_df["low"], ha_df["high"], colors=colors, linewidth=0.3
)
axes[0].set_ylabel("Price")
axes[0].set_title(f"{MARKET.id} Heikin Ashi Bars (1-min)")

axes[1].bar(range(len(ha_df)), ha_df["volume"], alpha=0.7, color="steelblue", width=1.0)
axes[1].set_ylabel("Volume")
axes[1].set_xlabel("Bar Index")

plt.tight_layout()
plt.show()

## 4. Range Bars

In [ ]:
range_df = store.create_chart(MARKET, "RANGE").read(DT_RANGE)

print(f"Shape: {range_df.shape}")
print(f"Time range: {range_df.index.min()} ~ {range_df.index.max()}")
print(f"NaN counts:\n{range_df.isna().sum()}")
display(range_df.describe())
display(range_df.head())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

colors = [
    "green" if c >= o else "red" for o, c in zip(range_df["open"], range_df["close"])
]
axes[0].bar(
    range(len(range_df)),
    range_df["close"] - range_df["open"],
    bottom=range_df["open"],
    color=colors,
    alpha=0.7,
    width=1.0,
)
axes[0].vlines(
    range(len(range_df)),
    range_df["low"],
    range_df["high"],
    colors=colors,
    linewidth=0.3,
)
axes[0].set_ylabel("Price")
axes[0].set_title(f"{MARKET.id} Range Bars (0.1%)")

axes[1].bar(
    range(len(range_df)), range_df["volume"], alpha=0.7, color="steelblue", width=1.0
)
axes[1].set_ylabel("Volume")

axes[2].plot(range_df["tick_count"].values, linewidth=0.5, color="purple")
axes[2].set_ylabel("Tick Count")
axes[2].set_xlabel("Bar Index")

plt.tight_layout()
plt.show()

## 5. Trade Aggression

In [ ]:
ta_df = store.create_single_market("TradeAggression", MARKET).read(DT_RANGE)

print(f"Shape: {ta_df.shape}")
print(f"Time range: {ta_df.index.min()} ~ {ta_df.index.max()}")
print(f"Columns: {list(ta_df.columns)}")
print(f"NaN counts:\n{ta_df.isna().sum()}")
display(ta_df.describe())
display(ta_df.head())

In [ ]:
# Imbalance (qty) per window
imb_cols = [c for c in ta_df.columns if c.startswith("imbalance_qty_")]
fig, axes = plt.subplots(len(imb_cols), 1, figsize=(15, 3 * len(imb_cols)), sharex=True)
if len(imb_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, imb_cols):
    ax.plot(ta_df.index, ta_df[col], linewidth=0.3, alpha=0.7)
    ax.set_ylabel(col)
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"{MARKET.id} Trade Imbalance (qty)")
plt.tight_layout()
plt.show()

In [ ]:
# Net flow per window
flow_cols = [c for c in ta_df.columns if c.startswith("net_flow_")]
fig, axes = plt.subplots(
    len(flow_cols), 1, figsize=(15, 3 * len(flow_cols)), sharex=True
)
if len(flow_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, flow_cols):
    ax.plot(ta_df.index, ta_df[col], linewidth=0.3, alpha=0.7)
    ax.set_ylabel(col)
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"{MARKET.id} Net Flow")
plt.tight_layout()
plt.show()

In [ ]:
# Buy/sell ratio per window
ratio_cols = [c for c in ta_df.columns if c.startswith("buy_sell_ratio_")]
fig, axes = plt.subplots(
    len(ratio_cols), 1, figsize=(15, 3 * len(ratio_cols)), sharex=True
)
if len(ratio_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, ratio_cols):
    ax.plot(ta_df.index, ta_df[col], linewidth=0.3, alpha=0.7)
    ax.set_ylabel(col)
    ax.axhline(y=1.0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"{MARKET.id} Buy/Sell Ratio")
plt.tight_layout()
plt.show()

In [ ]:
# VWAP per window
vwap_cols = [
    c
    for c in ta_df.columns
    if c.startswith("vwap_")
    and not c.startswith("vwap_buy")
    and not c.startswith("vwap_sell")
]
vwap_buy_cols = [c for c in ta_df.columns if c.startswith("vwap_buy_")]
vwap_sell_cols = [c for c in ta_df.columns if c.startswith("vwap_sell_")]

fig, axes = plt.subplots(
    len(vwap_cols), 1, figsize=(15, 3 * len(vwap_cols)), sharex=True
)
if len(vwap_cols) == 1:
    axes = [axes]
for ax, vcol, bcol, scol in zip(axes, vwap_cols, vwap_buy_cols, vwap_sell_cols):
    ax.plot(ta_df.index, ta_df[vcol], linewidth=0.3, alpha=0.7, label="vwap")
    ax.plot(
        ta_df.index,
        ta_df[bcol],
        linewidth=0.3,
        alpha=0.5,
        color="green",
        label="vwap_buy",
    )
    ax.plot(
        ta_df.index,
        ta_df[scol],
        linewidth=0.3,
        alpha=0.5,
        color="red",
        label="vwap_sell",
    )
    ax.set_ylabel(vcol)
    ax.legend(fontsize=8)
axes[0].set_title(f"{MARKET.id} VWAP")
plt.tight_layout()
plt.show()

In [ ]:
# Burst flags and trade activity per window
burst_cols = [c for c in ta_df.columns if c.startswith("burst_flag_")]
tps_cols = [c for c in ta_df.columns if c.startswith("trades_per_sec_")]

fig, axes = plt.subplots(
    len(burst_cols) + len(tps_cols),
    1,
    figsize=(15, 2.5 * (len(burst_cols) + len(tps_cols))),
    sharex=True,
)
if not isinstance(axes, list):
    axes = list(axes)
idx = 0
for col in tps_cols:
    axes[idx].plot(ta_df.index, ta_df[col], linewidth=0.3, alpha=0.7)
    axes[idx].set_ylabel(col, fontsize=8)
    idx += 1
for col in burst_cols:
    axes[idx].plot(ta_df.index, ta_df[col], linewidth=0.3, alpha=0.7, color="red")
    axes[idx].set_ylabel(col, fontsize=8)
    idx += 1
axes[0].set_title(f"{MARKET.id} Trade Activity & Burst Flags")
plt.tight_layout()
plt.show()

## 6. Data Consistency Check

In [ ]:
print("=" * 60)
print("Data Consistency Summary")
print("=" * 60)

for name, df in [
    ("Candle", candle_df),
    ("HeikinAshi", ha_df),
    ("Range", range_df),
    ("TradeAggression", ta_df),
]:
    print(f"\n--- {name} ---")
    print(f"  Rows:       {len(df):,}")
    print(f"  Columns:    {len(df.columns)}")
    print(f"  Time range: {df.index.min()} ~ {df.index.max()}")
    total_nan = df.isna().sum().sum()
    print(f"  Total NaN:  {total_nan:,} ({total_nan / df.size * 100:.2f}%)")
    dupes = df.index.duplicated().sum()
    print(f"  Duplicate timestamps: {dupes}")
    monotonic = df.index.is_monotonic_increasing
    print(f"  Monotonic index: {monotonic}")

# Sanity checks on chart bars
print("\n--- Chart Bar Sanity ---")
for name, df in [("Candle", candle_df), ("HeikinAshi", ha_df), ("Range", range_df)]:
    high_geq_low = (df["high"] >= df["low"]).all()
    high_geq_open = (df["high"] >= df["open"]).all()
    high_geq_close = (df["high"] >= df["close"]).all()
    low_leq_open = (df["low"] <= df["open"]).all()
    low_leq_close = (df["low"] <= df["close"]).all()
    positive_vol = (df["volume"] >= 0).all()
    print(
        f"  {name}: high>=low={high_geq_low}, high>=O/C={high_geq_open and high_geq_close}, "
        f"low<=O/C={low_leq_open and low_leq_close}, vol>=0={positive_vol}"
    )